# Banc JEPA physique — coordonnées + VISION (zéro upload)

Scripts récupérés depuis GitHub. Code changé → **re-rouler la cellule 1** (`git pull`).
Repo : https://github.com/frpatry/jepa-physics-bench

## 1. Récupérer / mettre à jour le code (re-rouler pour la dernière version)

In [ ]:
import os
if not os.path.exists('/content/jepa-physics-bench'):
    !git clone https://github.com/frpatry/jepa-physics-bench.git /content/jepa-physics-bench
!cd /content/jepa-physics-bench && git pull
%cd /content/jepa-physics-bench
!pip install -q lejepa || pip install -q git+https://github.com/rbalestr-lab/lejepa
print('OK — scripts à jour, lejepa installé')

## 2. Test API lejepa (~5 s)

In [ ]:
import torch, lejepa
t=lejepa.univariate.EppsPulley(t_max=3,n_points=17)
f=lejepa.multivariate.SlicingUnivariateTest(univariate_test=t,num_slices=1024)
z=torch.randn(128,128,requires_grad=True); f(z).backward()
print('lejepa OK | grad ok =', z.grad is not None)

## 3. COORDONNÉES — rebond + vent (5 configs), SIGReg-SSL (time)

In [ ]:
for bo,wi,tag,lab in [(0.0,0.0,'c0','calme'),(0.5,0.0,'cb','rebond'),(0.0,1.0,'cw','vent'),(0.5,1.0,'cbw','rebond+vent'),(0.9,2.0,'cx','fort+fort')]:
    print(f'\n##### {lab} (bounce={bo}, wind={wi}) #####')
    !python -u cl_sslbench.py --regs sigreg_off --readout time --reg_w 25 --bounce {bo} --wind {wi} --n_trains 1000 3000 --seeds 3 --steps 1500 --out runs_{tag}

## 4. Oracle (coordonnées) × configs

In [ ]:
for bo,wi,tag,lab in [(0.0,0.0,'c0','calme'),(0.5,0.0,'cb','rebond'),(0.0,1.0,'cw','vent'),(0.5,1.0,'cbw','rebond+vent'),(0.9,2.0,'cx','fort+fort')]:
    print(f'\n##### ORACLE {lab} #####')
    !python -u cl_sslbench.py --oracle --bounce {bo} --wind {wi} --n_trains 1000 3000

## 5. Tableau coordonnées

In [ ]:
import json, pandas as pd
CFG=[(0.0,0.0,'c0','calme'),(0.5,0.0,'cb','rebond'),(0.0,1.0,'cw','vent'),(0.5,1.0,'cbw','rebond+vent'),(0.9,2.0,'cx','fort+fort')]
rows=[]
for bo,wi,tag,lab in CFG:
    d=json.load(open(f'runs_{tag}/ssl_metrics.json'))
    for key,r in d['summary'].items():
        reg,n=key.split('|'); rows.append({'config':lab,'n':int(n),'sonde-g':round(r['acc'],2),'std':round(r.get('acc_std',0),2),'rang':round(r['rank'],1)})
df=pd.DataFrame(rows)
print(f"chance = {json.load(open('runs_c0/ssl_metrics.json'))['meta']['chance']:.0%}  (coordonnées, SIGReg time)")
for n in [3000,1000]:
    print(f'\n=== n={n} ==='); print(df[df.n==n][['config','sonde-g','std','rang']].to_string(index=False))

## 6. VISION — la balle dessinée en image 16×16 (façon V-JEPA)
Config chaotique fixe (**rebond + vent**), on monte le **bruit visuel** (pixels) : 0 → 0.25 → 0.5.
Question : l'image riche encaisse-t-elle mieux que les 32 coordonnées ? le bruit visuel fait-il moins mal que le vent ?

In [ ]:
for vn,tag in [(0.0,'v0'),(0.25,'v2'),(0.5,'v5')]:
    print(f'\n##### VISION rebond+vent  vnoise={vn} #####')
    !python -u cl_sslbench.py --regs sigreg_off --readout time --reg_w 25 --vision --grid 16 --bounce 0.5 --wind 1.0 --vnoise {vn} --n_trains 1000 3000 --seeds 3 --steps 1500 --out runs_{tag}

## 7. Oracle (vision) × bruit visuel

In [ ]:
for vn,tag in [(0.0,'v0'),(0.25,'v2'),(0.5,'v5')]:
    print(f'\n##### ORACLE VISION vnoise={vn} #####')
    !python -u cl_sslbench.py --oracle --vision --grid 16 --bounce 0.5 --wind 1.0 --vnoise {vn} --n_trains 1000 3000

## 8. Tableau vision

In [ ]:
import json, pandas as pd
VN=[(0.0,'v0'),(0.25,'v2'),(0.5,'v5')]
rows=[]
for vn,tag in VN:
    d=json.load(open(f'runs_{tag}/ssl_metrics.json'))
    for key,r in d['summary'].items():
        reg,n=key.split('|'); rows.append({'vnoise':vn,'n':int(n),'sonde-g':round(r['acc'],2),'std':round(r.get('acc_std',0),2),'rang':round(r['rank'],1)})
df=pd.DataFrame(rows)
print('VISION 16x16, rebond+vent | chance = 17%')
for n in [3000,1000]:
    print(f'\n=== n={n} ==='); print(df[df.n==n][['vnoise','sonde-g','std','rang']].to_string(index=False))
print('\n(reference COORDONNEES rebond+vent etait ~0.63 a n=3000)')